# Load Dataset And Show Basic Information

In [126]:
import numpy as np
import pandas as pd

In [127]:
def verbose_dataset_info(interactions):
    print("DATASET INFORMATION:")
    users_size = interactions['uid'].nunique()
    print(f"\tNumber of users: {users_size}, index: {min(interactions['uid'])}-{max(interactions['uid'])}")
    items_size = interactions['iid'].nunique()
    print(f"\tNumber of items: {items_size}, index: {min(interactions['iid'])}-{max(interactions['iid'])}")

    interaction_size = len(interactions)
    print(f"\tNumber of ratings: {interaction_size}")
    sparsity = (users_size * items_size - interaction_size) / (users_size * items_size)
    print(f"\tSparsity: {sparsity} = {users_size * items_size - interaction_size} / {users_size * items_size}")

In [113]:
def load_filmtrust(verbose=True):
    interactions = pd.read_csv('../raw_datasets/filmtrust-ratings.txt', header=None, sep='\s+',
                               names=['uid', 'iid', 'rating'], engine='python')

    rng = np.random.default_rng(42)
    n = len(interactions)
    base = rng.integers(low=0, high=10 ** 9, size=n, dtype=np.int64)
    offset = interactions.groupby('uid').cumcount().astype(np.int64)
    interactions['timestamp'] = (base * np.int64(10 ** 6) + offset).astype(np.int64)

    interactions = interactions[['uid', 'iid', 'rating', 'timestamp']]

    if verbose:
        verbose_dataset_info(interactions)

    return interactions

In [114]:
def load_lastfm_2k(verbose=True):
    interactions = pd.read_csv('../raw_datasets/last-fm-2k-ratings.dat', sep=",", header=None,
                               names=['uid', 'iid', 'rating', 'timestamp'], engine='python')

    interactions['uid'] = pd.factorize(interactions['uid'])[0] + 1
    interactions['iid'] = pd.factorize(interactions['iid'])[0] + 1

    if verbose:
        verbose_dataset_info(interactions)

    return interactions

In [128]:
def load_amazon_video(verbose=True):
    interactions = pd.read_csv('../raw_datasets/amazon-video-ratings.dat', sep=",", header=None,
                               names=['uid', 'iid', 'rating', 'timestamp'], engine='python')

    interactions['uid'] = pd.factorize(interactions['uid'])[0] + 1
    interactions['iid'] = pd.factorize(interactions['iid'])[0] + 1

    if verbose:
        verbose_dataset_info(interactions)

    return interactions

In [116]:
def load_qb_article(verbose=True):
    interactions = pd.read_csv('../raw_datasets/QB-article.csv', sep=",", usecols=['user_id', 'item_id'],
                               engine='python')
    interactions['rating'] = 1
    interactions['timestamp'] = interactions.groupby('user_id').cumcount() + 1

    interactions['uid'] = pd.factorize(interactions['user_id'])[0] + 1
    interactions['iid'] = pd.factorize(interactions['item_id'])[0] + 1

    interactions = interactions[['uid', 'iid', 'rating', 'timestamp']]

    if verbose:
        verbose_dataset_info(interactions)

    return interactions

In [129]:
def load_dataset(dataset_name='filmtrust', verbose=True):
    if dataset_name == 'filmtrust':
        return load_filmtrust(verbose=verbose)
    elif dataset_name == 'lastfm_2k':
        return load_lastfm_2k(verbose=verbose)
    elif dataset_name == 'amazon_video':
        return load_amazon_video(verbose=verbose)
    elif dataset_name == 'qb_article':
        return load_qb_article(verbose=verbose)
    else:
        raise ValueError(f'Dataset {dataset_name} not supported')

In [130]:
verbose = True  # Whether to output detailed information.
dataset = 'amazon_video'  # Select the dataset to process, select from ['filmtrust', 'lastfm_2k', 'amazon_video', 'qb_article'].

In [131]:
interactions_original = load_dataset(dataset, verbose)

DATASET INFORMATION:
	Number of users: 8072, index: 1-8072
	Number of items: 11830, index: 1-11830
	Number of ratings: 63836
	Sparsity: 0.9993315025296423 = 95427924 / 95491760


# Data Cleaning


In [132]:
def data_cleaning(interactions, user_threshold=5, verbose=True):
    user_counts = interactions['uid'].value_counts()
    keep_uids = set(user_counts[user_counts >= user_threshold].index)
    interactions = interactions[interactions['uid'].isin(keep_uids)]

    if verbose:
        print(f"Keep users with >= {user_threshold} interactions: {len(keep_uids)} users")
        print(f"Remaining interactions: {len(interactions)}")

    keep_iids = set(interactions['iid'].unique())
    interactions = interactions[interactions['iid'].isin(keep_iids)]

    if verbose:
        print(f"Remaining items: {len(keep_iids)}")

    user_index_mapping = {old_idx: new_idx for new_idx, old_idx in enumerate(sorted(interactions['uid'].unique()))}
    item_index_mapping = {old_idx: new_idx for new_idx, old_idx in enumerate(sorted(interactions['iid'].unique()))}
    interactions.loc[:, 'uid'] = interactions['uid'].map(user_index_mapping)
    interactions.loc[:, 'iid'] = interactions['iid'].map(item_index_mapping)

    interactions = interactions.sort_values(by=['uid', 'iid']).reset_index(drop=True)

    if verbose:
        verbose_dataset_info(interactions)

    return interactions


In [133]:
interactions = data_cleaning(interactions_original, 5)

Keep users with >= 5 interactions: 8072 users
Remaining interactions: 63836
Remaining items: 11830
DATASET INFORMATION:
	Number of users: 8072, index: 0-8071
	Number of items: 11830, index: 0-11829
	Number of ratings: 63836
	Sparsity: 0.9993315025296423 = 95427924 / 95491760


# Negative Sampling And Generate Interaction For Training / Testing

In [134]:
import random
import os
import builtins

In [135]:
num_neg = 99  # The number of negative samples, each user has one positive sample and 99 negative samples.
random.seed(42)  # This experiment was conducted with a seed of 42.

In [136]:
def generate_train_test_interaction(interactions, dataset, num_neg=99, verbose=True):
    interactions['latest'] = interactions.groupby('uid')['timestamp'].rank(method='first', ascending=False)
    test = interactions[interactions['latest'] == 1]
    train = interactions[interactions['latest'] > 1]
    train_interaction = train[['uid', 'iid']]
    test_interaction = test[['uid', 'iid']]
    if verbose:
        print(f"Number of users in test {test['uid'].nunique()}, in train {train['uid'].nunique()}")

    save_path = f"./{dataset}/preprocess/"
    os.makedirs(save_path, exist_ok=True)
    train_interaction.to_csv(os.path.join(save_path, 'train_interactions.csv'), header=False, index=False)

    item_pool = set(interactions['iid'].unique())
    interact_status = interactions.groupby('uid')['iid'].apply(set).reset_index().rename(
        columns={'iid': 'interacted_items'})
    interact_status['negative_items'] = interact_status['interacted_items'].apply(
        lambda x: set(random.sample(list(item_pool), (num_neg + len(x)))) - x)
    interact_status['negative_samples'] = interact_status['negative_items'].apply(
        lambda x: random.sample(list(x), num_neg))
    test_interaction = pd.merge(test_interaction, interact_status[['uid', 'negative_samples']], on='uid')
    with open(os.path.join(save_path, 'test_neg_interactions'), 'w') as f:
        for row in test_interaction.itertuples():
            t_rating = (builtins.int(row.uid), builtins.int(row.iid))
            f.write(str(t_rating))
            for i in range(len(row.negative_samples)):
                f.write('\t')
                f.write(str(row.negative_samples[i]))
            f.write('\n')

    return train_interaction, test_interaction

In [137]:
train_interaction, test_interaction = generate_train_test_interaction(interactions, dataset, num_neg, verbose)

Number of users in test 8072, in train 8072
